In [1]:
# =========================================
# DHAIRYA LUNIA - 20BCE2178
# =========================================

In [2]:
# =========================================
# 1. IMPORTS
# =========================================
import os
import zipfile
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:

# =========================================
# 2. UNZIP DATA
# =========================================
TRAIN_ZIP = "/kaggle/input/competitions/dogs-vs-cats-redux-kernels-edition/train.zip"
TEST_ZIP  = "/kaggle/input/competitions/dogs-vs-cats-redux-kernels-edition/test.zip"

with zipfile.ZipFile(TRAIN_ZIP, 'r') as z:
    z.extractall('/kaggle/working/')
with zipfile.ZipFile(TEST_ZIP, 'r') as z:
    z.extractall('/kaggle/working/')

TRAIN_DIR = "/kaggle/working/train"
TEST_DIR  = "/kaggle/working/test"

In [4]:

# =========================================
# 3. DATASET
# =========================================
class DogsCatsDataset(Dataset):
    def __init__(self, file_paths, labels=None, transform=None):
        self.file_paths = file_paths
        self.labels     = labels
        self.transform  = transform

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        img = Image.open(self.file_paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        if self.labels is not None:
            return img, torch.tensor(self.labels[idx], dtype=torch.float32)
        return img

In [5]:
# =========================================
# 4. LOAD & SPLIT DATA
# =========================================
train_files = os.listdir(TRAIN_DIR)
file_paths, labels = [], []
 
for file in train_files:
    file_paths.append(os.path.join(TRAIN_DIR, file))
    labels.append(1 if "dog" in file else 0)
 
train_paths, val_paths, train_labels, val_labels = train_test_split(
    file_paths, labels, test_size=0.2, random_state=42, stratify=labels
)
print(f"Train: {len(train_paths)} | Val: {len(val_paths)}")

Train: 20000 | Val: 5000


In [6]:

# =========================================
# 5. TRANSFORMS
# =========================================
IMG_SIZE = 380

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.65, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomRotation(25),                              
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.12))      
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [7]:
# =========================================
# 6. DATALOADERS
# =========================================
BATCH_SIZE  = 24
ACCUM_STEPS = 6   # effective batch = 24 × 6 = 144

train_dataset = DogsCatsDataset(train_paths, train_labels, train_transform)
val_dataset   = DogsCatsDataset(val_paths,   val_labels,   val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)


In [8]:
# =========================================
# 7. MIXUP
# =========================================
def mixup_data(x, y, alpha=0.4):
    lam   = np.random.beta(alpha, alpha)
    idx   = torch.randperm(x.size(0)).to(device)
    x_mix = lam * x + (1 - lam) * x[idx]
    return x_mix, y, y[idx], lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [9]:
# =========================================
# 8. MODEL
# =========================================
def build_model():
    weights = EfficientNet_B3_Weights.DEFAULT
    model   = efficientnet_b3(weights=weights)

    # Freeze entire backbone first
    for param in model.parameters():
        param.requires_grad = False

    for param in model.features[-2:].parameters():
        param.requires_grad = True

    in_features = model.classifier[1].in_features  # 1536 for B3

    # NEW: BatchNorm added before dropout
    model.classifier[1] = nn.Sequential(
        nn.BatchNorm1d(in_features),   # normalize incoming 1536 features
        nn.Dropout(0.3),               # regularization
        nn.Linear(in_features, 1)      # binary output
    )

    # Small weight init — prevents large early logits
    nn.init.xavier_uniform_(model.classifier[1][2].weight)
    nn.init.zeros_(model.classifier[1][2].bias)

    return model.to(device)

model = build_model()

In [10]:
# =========================================
# 9. LOSS + OPTIMIZER + SCHEDULER + SCALER
# =========================================
criterion = nn.BCEWithLogitsLoss()
scaler    = torch.amp.GradScaler('cuda')

EPOCHS        = 5
WARMUP_EPOCHS = 1

for param in model.features[-4:].parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW([
    {'params': model.classifier.parameters(),    'lr': 3e-4, 'weight_decay': 1e-4},
    {'params': model.features[-4:].parameters(), 'lr': 5e-5, 'weight_decay': 1e-4},
])

def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    return 1.0

warmup_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS - WARMUP_EPOCHS, eta_min=1e-6
)

# =========================================
# 10. PROGRESSIVE UNFREEZING
# =========================================
def unfreeze_phase(phase):
    if phase == 2:
        print("\n>>> Unfreezing full backbone")
        for param in model.parameters():
            param.requires_grad = True
        optimizer.add_param_group(
            {'params': model.features[:-4].parameters(), 'lr': 1e-5, 'weight_decay': 1e-4}
        )

# =========================================
# 11. TRAINING LOOP
# =========================================
def train_model(epochs=EPOCHS):
    best_val_loss = float("inf")
    patience      = 3   
    no_improve    = 0

    for epoch in range(1, epochs + 1):

        if epoch == 3:
            unfreeze_phase(phase=2)

        # ---- TRAIN ----
        model.train()
        train_loss = 0.0
        optimizer.zero_grad()

        for step, (images, batch_labels) in enumerate(train_loader):
            images       = images.to(device)
            batch_labels = batch_labels.to(device).unsqueeze(1)

            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss    = criterion(outputs, batch_labels)

            loss = loss / ACCUM_STEPS
            scaler.scale(loss).backward()

            if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            train_loss += loss.item() * ACCUM_STEPS

        # ---- VALIDATE ----
        model.eval()
        val_loss = 0.0
        correct  = 0
        total    = 0

        with torch.no_grad():
            for images, batch_labels in val_loader:
                images       = images.to(device)
                batch_labels = batch_labels.to(device).unsqueeze(1)

                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                    loss    = criterion(outputs, batch_labels)

                val_loss += loss.item()
                preds     = (torch.sigmoid(outputs) > 0.5).float()
                correct  += (preds == batch_labels).sum().item()
                total    += batch_labels.size(0)

        if epoch <= WARMUP_EPOCHS:
            warmup_scheduler.step()
        else:
            cosine_scheduler.step()

        avg_train = train_loss / len(train_loader)
        avg_val   = val_loss   / len(val_loader)
        accuracy  = 100 * correct / total
        lr_now    = optimizer.param_groups[0]['lr']

        print(f"Epoch {epoch:02d}/{epochs} | "
              f"Train: {avg_train:.4f} | "
              f"Val: {avg_val:.4f} | "
              f"Acc: {accuracy:.2f}% | "
              f"LR: {lr_now:.2e}")

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            no_improve    = 0
            torch.save(model.state_dict(), "best_model.pth")
            print(f"  ✅ Best model saved (val loss: {best_val_loss:.4f})")
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"\n⏹ Early stopping at epoch {epoch}")
                break

    print(f"\nBest Val Loss: {best_val_loss:.4f}")

train_model(epochs=EPOCHS)
model.load_state_dict(torch.load("best_model.pth"))
print("Best model loaded.")

Epoch 01/5 | Train: 0.2148 | Val: 0.0284 | Acc: 98.88% | LR: 3.00e-04
  ✅ Best model saved (val loss: 0.0284)
Epoch 02/5 | Train: 0.0870 | Val: 0.0205 | Acc: 99.34% | LR: 2.56e-04
  ✅ Best model saved (val loss: 0.0205)

>>> Unfreezing full backbone
Epoch 03/5 | Train: 0.0696 | Val: 0.0168 | Acc: 99.46% | LR: 1.50e-04
  ✅ Best model saved (val loss: 0.0168)
Epoch 04/5 | Train: 0.0633 | Val: 0.0151 | Acc: 99.46% | LR: 4.48e-05
  ✅ Best model saved (val loss: 0.0151)
Epoch 05/5 | Train: 0.0566 | Val: 0.0162 | Acc: 99.42% | LR: 1.00e-06

Best Val Loss: 0.0151
Best model loaded.


In [11]:
# =========================================
# 12. TEST 
# =========================================
test_files = sorted(os.listdir(TEST_DIR), key=lambda x: int(x.split('.')[0]))
test_paths = [os.path.join(TEST_DIR, f) for f in test_files]

test_dataset = DogsCatsDataset(test_paths, transform=val_transform)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model.eval()
preds = []

with torch.no_grad():
    for images in test_loader:
        images  = images.to(device)
        with torch.amp.autocast('cuda'):
            outputs = model(images)
        probs = torch.sigmoid(outputs)
        preds.extend(probs.cpu().numpy())

final_predictions = np.array(preds).reshape(-1)

# =========================================
# 13. SUBMISSION
# =========================================
ids = [int(f.split('.')[0]) for f in test_files]
submission = pd.DataFrame({"id": ids, "label": final_predictions})
submission.to_csv("final_submission.csv", index=False)
print("\n✅ submission.csv created!")
print(submission.describe())


✅ submission.csv created!
                 id         label
count  12500.000000  12500.000000
mean    6250.500000      0.500488
std     3608.583517      0.496094
min        1.000000      0.000000
25%     3125.750000      0.000043
50%     6250.500000      0.635986
75%     9375.250000      0.999512
max    12500.000000      1.000000
